# Tier Fallback Test — Free → Hybrid → Cloud → Local

Validates that LiteLLM's cross-tier fallback chain works end-to-end.

## Strategy

**Phase 1 — Happy path:** confirms every real tier alias responds normally.

**Phase 2 — Fallback mechanism:** registers temporary *sentinel* model groups that
point to a dead endpoint (`http://0.0.0.0:1`, connection refused instantly).  
Per-request `fallbacks` are passed in each request body — no global config changes needed:
```
test/tier-free-{RUN_ID}   (dead sentinel)
  → test/tier-hybrid-{RUN_ID}  (dead sentinel)
    → test/tier-cloud-{RUN_ID}   (dead sentinel)
      → test/tier-local-{RUN_ID}  (real — local Ollama llama3.2:3b)
```
Detection uses the `x-litellm-model-group` response header (which group actually served)
and `x-litellm-attempted-fallbacks` (confirms the fallback path was taken).

**Phase 3 — Config audit:** detects whether the running LiteLLM instance has stale
fallback config in its DB that overrides `litellm_config.yaml`.

**Cleanup:** all sentinel model groups are deleted in a `try/finally` block.

---
> **Run order:** execute cells top-to-bottom.  
> The cleanup cell is safe to re-run if anything goes wrong midway.

In [1]:
import os, time, uuid, traceback
import requests

LITELLM_URL = os.environ.get('LITELLM_BASE_URL', 'http://litellm:4000')
LITELLM_KEY = os.environ.get('LITELLM_API_KEY', '')
OLLAMA_URL  = os.environ.get('OLLAMA_BASE_URL',  'http://ollama:11434')

H = {
    'Authorization': f'Bearer {LITELLM_KEY}',
    'Content-Type':  'application/json',
}

RUN_ID = uuid.uuid4().hex[:8]   # unique per run — prevents cross-run collisions
PROMPT = 'Reply with the single word: PONG'

_registered: list[dict] = []   # [{name, model_id}, ...] — populated by register_*
results:      list[dict] = []   # accumulated test results for the summary cell

if not LITELLM_KEY:
    raise RuntimeError('LITELLM_API_KEY env var is not set')

print(f'Run ID  : {RUN_ID}')
print(f'LiteLLM : {LITELLM_URL}')
print(f'Ollama  : {OLLAMA_URL}')
print(f'API key : ...{LITELLM_KEY[-6:]}')

Run ID  : 9151c444
LiteLLM : http://litellm:4000
Ollama  : http://ollama:11434
API key : ...ey-123


In [2]:
# ── Verified API behaviour (probed against live instance) ─────────────────────
#
# POST /model/new   → returns top-level 'model_id' (UUID) as the DB primary key
# POST /model/delete → requires body {'id': '<model_id UUID>'}
# x-litellm-model-group  header → group alias that actually served the request
# x-litellm-attempted-fallbacks header → int; >0 means fallback path was taken
# response body 'model' field → underlying provider model string (e.g. ollama_chat/...)
#
# NOTE: 'x-litellm-model-used' does NOT exist in this version (1.81.x).
# ─────────────────────────────────────────────────────────────────────────────

DEAD_ENDPOINT = 'http://0.0.0.0:1'   # connection refused in <1 ms — guaranteed failure
PROVIDER_LIMIT_CODES = {401, 429}     # provider-side quota/auth — expected, not test bugs


def extract_content(r: requests.Response) -> str:
    """
    Safely extract assistant message text.
    Handles content=null from reasoning models (e.g. step-3.5-flash) which
    exhaust max_tokens on thinking and set content=null / reasoning_content=...
    Never raises.
    """
    try:
        msg = r.json()['choices'][0]['message']
        content = msg.get('content')
        if content is not None:
            return content
        # Reasoning model: content is null, extract from reasoning_content
        reasoning = msg.get('reasoning_content') or msg.get('reasoning') or ''
        snippet = reasoning[:60].replace('\n', ' ') if reasoning else ''
        return f'[content=null; reasoning: {snippet!r}]' if snippet else '[content=null]'
    except Exception as exc:
        return f'[parse error: {exc}]'


def _register(group_name: str, litellm_model: str,
               api_base: str, api_key: str, timeout: int) -> str:
    """Core registration helper. Returns the DB model_id UUID."""
    payload = {
        'model_name': group_name,
        'litellm_params': {
            'model':    litellm_model,
            'api_base': api_base,
            'api_key':  api_key,
            'timeout':  timeout,
        },
        'model_info': {'description': f'tier-fallback-test run={RUN_ID}'},
    }
    r = requests.post(f'{LITELLM_URL}/model/new', headers=H, json=payload, timeout=10)
    r.raise_for_status()
    data = r.json()
    # 'model_id' is the top-level DB primary key returned by /model/new
    model_id = data.get('model_id') or data.get('model_info', {}).get('id')
    if not model_id:
        raise ValueError(f'/model/new response had no model_id: {data}')
    _registered.append({'name': group_name, 'model_id': model_id})
    return model_id


def register_sentinel(group_name: str) -> str:
    """Register a dead-endpoint sentinel under group_name. Always fails instantly."""
    mid = _register(group_name,
                    litellm_model='openai/test-sentinel-do-not-use',
                    api_base=DEAD_ENDPOINT,
                    api_key='sk-invalid',
                    timeout=2)
    print(f'  + sentinel {group_name!r}  model_id={mid}')
    return mid


def register_real(group_name: str, litellm_model: str,
                  api_base: str, api_key: str = 'ollama') -> str:
    """Register a live model under group_name."""
    mid = _register(group_name, litellm_model, api_base, api_key, timeout=120)
    print(f'  + real    {group_name!r}  model_id={mid}')
    return mid


def delete_model(entry: dict) -> bool:
    """Delete by DB model_id UUID. Returns True on success."""
    try:
        r = requests.post(
            f'{LITELLM_URL}/model/delete',
            headers=H, json={'id': entry['model_id']}, timeout=10,
        )
        r.raise_for_status()
        return True
    except Exception as e:
        print(f'  ⚠  delete {entry["name"]} (id={entry["model_id"]}): {e}')
        return False


def chat(model: str, fallbacks: list[str] | None = None,
         request_timeout: int = 300) -> requests.Response:
    """Send a minimal chat-completion to the proxy."""
    body: dict = {
        'model':    model,
        'messages': [{'role': 'user', 'content': PROMPT}],
        'max_tokens': 50,   # 50 so reasoning models have budget for content
    }
    if fallbacks:
        body['fallbacks'] = fallbacks
    return requests.post(
        f'{LITELLM_URL}/v1/chat/completions',
        headers=H, json=body, timeout=request_timeout,
    )


def serving_group(r: requests.Response) -> str:
    """
    Return the model GROUP that actually served the request.
    Uses x-litellm-model-group header (confirmed present in LiteLLM 1.81.x).
    Falls back to body 'model' field (underlying provider model string).
    """
    return r.headers.get('x-litellm-model-group', '') or r.json().get('model', 'unknown')


def fallback_count(r: requests.Response) -> int:
    """Number of fallback hops taken (from x-litellm-attempted-fallbacks header)."""
    try:
        return int(r.headers.get('x-litellm-attempted-fallbacks', '0'))
    except ValueError:
        return 0


def run_test(name: str, primary: str, fallback_chain: list[str],
             expected_group: str) -> dict:
    """
    Execute one fallback test.

    primary        — model group to call first (sentinel / dead)
    fallback_chain — ordered list of fallback groups passed in the request body
    expected_group — model group name expected to have served the final response

    PASS criteria:
      1. HTTP 200
      2. x-litellm-model-group == expected_group
      3. x-litellm-attempted-fallbacks >= 1
    """
    print(f'  primary   : {primary}')
    print(f'  fallbacks : {fallback_chain}')
    print(f'  expecting : {expected_group}')
    t0 = time.perf_counter()
    result = {'test': name, 'status': 'FAIL', 'group': '', 'fb_count': '', 'elapsed': ''}
    try:
        r = chat(primary, fallbacks=fallback_chain, request_timeout=120)
        elapsed = f'{time.perf_counter() - t0:.1f}s'
        result['elapsed'] = elapsed

        if r.ok:
            grp        = serving_group(r)
            fbs        = fallback_count(r)
            body_model = r.json().get('model', '')
            answer     = extract_content(r).strip()[:40]   # null-safe

            group_ok    = (grp == expected_group)
            fallback_ok = (fbs >= 1)
            passed      = group_ok and fallback_ok

            result.update(status='PASS' if passed else 'WARN',
                          group=grp, fb_count=str(fbs))
            icon = '✅' if passed else '⚠️ '
            print(f'  {icon} HTTP 200 in {elapsed}')
            print(f'     x-litellm-model-group       : {grp}')
            print(f'     x-litellm-attempted-fallbacks: {fbs}')
            print(f'     response body model          : {body_model}')
            print(f'     answer                       : {answer!r}')
            if not group_ok:
                print(f'     ⚠  serving group {grp!r} ≠ expected {expected_group!r}')
            if not fallback_ok:
                print(f'     ⚠  attempted-fallbacks={fbs} — fallback path not confirmed')
        else:
            result.update(status='FAIL', group=f'HTTP {r.status_code}')
            try:
                err = r.json().get('error', {}).get('message', r.text)[:400]
            except Exception:
                err = r.text[:400]
            print(f'  ❌ HTTP {r.status_code} in {elapsed}')
            print(f'     {err}')
    except Exception as exc:
        elapsed = f'{time.perf_counter() - t0:.1f}s'
        result.update(status='ERROR', group=str(exc)[:80], elapsed=elapsed)
        print(f'  ❌ Exception in {elapsed}: {exc}')
        traceback.print_exc()
    return result


print('Helpers loaded.')


Helpers loaded.


---
## Pre-flight — verify services are reachable

In [3]:
all_ok = True

# ── Service reachability ──────────────────────────────────────────────────────
for label, url, needs_auth in [
    ('LiteLLM /health/liveliness', f'{LITELLM_URL}/health/liveliness', True),
    ('LiteLLM /v1/models',         f'{LITELLM_URL}/v1/models',         True),
    ('Ollama  /api/version',       f'{OLLAMA_URL}/api/version',        False),
]:
    try:
        r = requests.get(url, headers=(H if needs_auth else {}), timeout=5)
        ok = r.ok
        print(f'  {"✅" if ok else "❌"}  {label} → HTTP {r.status_code}')
        all_ok = all_ok and ok
    except Exception as e:
        print(f'  ❌  {label} → {e}')
        all_ok = False

# ── Confirm llama3.2:3b is pulled in Ollama (used as the real model in Phase 2) ──
try:
    tags = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5).json().get('models', [])
    names = [m['name'] for m in tags]
    has_model = any('llama3.2' in n for n in names)
    print(f'  {"✅" if has_model else "❌"}  ollama llama3.2:3b available  '
          f'(models: {", ".join(names) or "none"})')
    if not has_model:
        print('     → fix: docker exec ollama ollama pull llama3.2:3b')
        all_ok = False
except Exception as e:
    print(f'  ❌  Ollama model list: {e}')
    all_ok = False

if not all_ok:
    raise RuntimeError('Pre-flight failed — resolve the issues above before continuing')

print('\nPre-flight passed ✅')

  ✅  LiteLLM /health/liveliness → HTTP 200
  ✅  LiteLLM /v1/models → HTTP 200
  ✅  Ollama  /api/version → HTTP 200
  ✅  ollama llama3.2:3b available  (models: qwen2.5-coder:7b, deepseek-r1:7b, mistral:7b-instruct-q4_0, llama3.1:8b-instruct-q4_0, llama3.2:3b, mistral:latest)

Pre-flight passed ✅


---
## Phase 1 — Happy-path smoke test

Call each real tier alias and confirm it returns a valid response.  
The `x-litellm-model-group` header is shown for each — on a direct (non-fallback)
call it echoes the requested group name back.  

A failure in `cloud/fast` is expected if Gemini free-tier quota is exhausted;
Phase 2 tests are unaffected (they use per-request fallbacks + local Ollama only).

In [4]:
SMOKE_TIERS = [
    ('local/fast',  'local'),
    ('free/fast',   'free'),
    ('hybrid/fast', 'hybrid'),
    ('cloud/fast',  'cloud'),
]

print('── Phase 1: Happy-path smoke test ───────────────────────────────────────')
print(f'{"Tier":<22} {"Status":<8} {"Group served":<30} {"Time":<7} Answer')
print('─' * 95)

for tier, label in SMOKE_TIERS:
    # Pre-initialise all display variables BEFORE the try block so that
    # the except clause never writes over a successfully-captured value.
    elapsed = '?'; grp = ''; ans = ''; status = 'ERROR'; icon = '❌'
    t0 = time.perf_counter()
    try:
        r = chat(tier, request_timeout=120)
        elapsed = f'{time.perf_counter() - t0:.1f}s'
        if r.ok:
            grp    = serving_group(r)
            ans    = extract_content(r).strip()[:25]   # null-safe for reasoning models
            status = 'PASS'
            icon   = '✅'
            results.append({'test': f'Phase1/{label}', 'status': 'PASS',
                            'group': grp, 'fb_count': '0', 'elapsed': elapsed})
        else:
            grp = f'HTTP {r.status_code}'
            if r.status_code in PROVIDER_LIMIT_CODES:
                # 401 / 429 = provider quota / auth issue; expected when keys are
                # exhausted or invalid. Flag as WARN (not a test framework bug).
                status = 'WARN'
                icon   = '⚠️ '
                try:
                    err_msg = r.json().get('error', {}).get('message', '')[:60]
                except Exception:
                    err_msg = r.text[:60]
                ans = err_msg
            else:
                status = 'FAIL'
                icon   = '❌'
            results.append({'test': f'Phase1/{label}', 'status': status,
                            'group': grp, 'fb_count': '', 'elapsed': elapsed})
    except Exception as e:
        elapsed = f'{time.perf_counter() - t0:.1f}s'
        if not grp:
            grp = type(e).__name__
        ans    = str(e)[:25]
        status = 'ERROR'
        icon   = '❌'
        results.append({'test': f'Phase1/{label}', 'status': 'ERROR',
                        'group': grp, 'fb_count': '', 'elapsed': elapsed})
    print(f'{tier:<22} {icon} {status:<6} {grp:<30} {elapsed:<7} {ans!r}')


── Phase 1: Happy-path smoke test ───────────────────────────────────────
Tier                   Status   Group served                   Time    Answer
───────────────────────────────────────────────────────────────────────────────────────────────


local/fast             ✅ PASS   local/fast                     6.7s    'PONG'


free/fast              ✅ PASS   free/fast                      1.9s    '[content=null; reasoning:'


hybrid/fast            ✅ PASS   hybrid/fast                    0.2s    'PONG'


cloud/fast             ✅ PASS   cloud/fast                     1.4s    'PONG'


---
## Phase 2 — Fallback mechanism tests

### How sentinel injection works

1. Register **dead** sentinel groups pointing to `http://0.0.0.0:1` (connection refused <1 ms)
2. Register one **real** group pointing to local Ollama `llama3.2:3b`
3. Pass `fallbacks=[...]` in each request body — LiteLLM tries groups in order
4. Assert via `x-litellm-model-group` header that the real group served the response
5. Assert `x-litellm-attempted-fallbacks >= 1` to confirm the fallback path fired

This is self-contained: no global router config is modified.

In [5]:
T_FREE   = f'test/tier-free-{RUN_ID}'
T_HYBRID = f'test/tier-hybrid-{RUN_ID}'
T_CLOUD  = f'test/tier-cloud-{RUN_ID}'
T_LOCAL  = f'test/tier-local-{RUN_ID}'

# Single-hop tests re-use T_LOCAL as the live target, but also need live aliases
# for the 'hybrid' and 'cloud' positions so each hop test has a real endpoint.
T_HYBRID_LIVE = f'test/tier-hybrid-live-{RUN_ID}'
T_CLOUD_LIVE  = f'test/tier-cloud-live-{RUN_ID}'

print(f'── Registering test model groups (run={RUN_ID}) ──────────────────────────')
print('\nDead sentinels:')
register_sentinel(T_FREE)
register_sentinel(T_HYBRID)
register_sentinel(T_CLOUD)

print('\nReal endpoints (all wired to local Ollama llama3.2:3b):')
register_real(T_LOCAL,       'ollama_chat/llama3.2:3b', api_base=OLLAMA_URL)
register_real(T_HYBRID_LIVE, 'ollama_chat/llama3.2:3b', api_base=OLLAMA_URL)
register_real(T_CLOUD_LIVE,  'ollama_chat/llama3.2:3b', api_base=OLLAMA_URL)

print(f'\nTotal registered for cleanup this run: {len(_registered)}')

── Registering test model groups (run=9151c444) ──────────────────────────

Dead sentinels:
  + sentinel 'test/tier-free-9151c444'  model_id=689799ed-db17-47fd-bba7-a708ae1c330c
  + sentinel 'test/tier-hybrid-9151c444'  model_id=7e1b0083-99a3-4a47-97d4-1381c6847861


  + sentinel 'test/tier-cloud-9151c444'  model_id=236b371d-4e25-4282-af35-962d4f498686

Real endpoints (all wired to local Ollama llama3.2:3b):
  + real    'test/tier-local-9151c444'  model_id=26717e31-2404-4893-8f21-7eb6810fbc7c
  + real    'test/tier-hybrid-live-9151c444'  model_id=e9c5cd01-72ea-4c45-acc6-0b5025b3535c


  + real    'test/tier-cloud-live-9151c444'  model_id=bf1f6061-df17-4d46-8ecc-da11afebf3eb

Total registered for cleanup this run: 6


### Test A — Single hop: Free → Hybrid

In [6]:
print('── Test A: Free → Hybrid ────────────────────────────────────────────────')
res = run_test(
    name           = 'A: Free→Hybrid',
    primary        = T_FREE,
    fallback_chain = [T_HYBRID_LIVE],
    expected_group = T_HYBRID_LIVE,
)
results.append(res)

── Test A: Free → Hybrid ────────────────────────────────────────────────
  primary   : test/tier-free-9151c444
  fallbacks : ['test/tier-hybrid-live-9151c444']
  expecting : test/tier-hybrid-live-9151c444


  ✅ HTTP 200 in 7.1s
     x-litellm-model-group       : test/tier-hybrid-live-9151c444
     x-litellm-attempted-fallbacks: 1
     response body model          : ollama_chat/llama3.2:3b
     answer                       : 'PONG'


### Test B — Single hop: Hybrid → Cloud

In [7]:
print('── Test B: Hybrid → Cloud ───────────────────────────────────────────────')
res = run_test(
    name           = 'B: Hybrid→Cloud',
    primary        = T_HYBRID,
    fallback_chain = [T_CLOUD_LIVE],
    expected_group = T_CLOUD_LIVE,
)
results.append(res)

── Test B: Hybrid → Cloud ───────────────────────────────────────────────
  primary   : test/tier-hybrid-9151c444
  fallbacks : ['test/tier-cloud-live-9151c444']
  expecting : test/tier-cloud-live-9151c444


  ✅ HTTP 200 in 7.4s
     x-litellm-model-group       : test/tier-cloud-live-9151c444
     x-litellm-attempted-fallbacks: 1
     response body model          : ollama_chat/llama3.2:3b
     answer                       : 'PONG'


### Test C — Single hop: Cloud → Local

In [8]:
print('── Test C: Cloud → Local ────────────────────────────────────────────────')
res = run_test(
    name           = 'C: Cloud→Local',
    primary        = T_CLOUD,
    fallback_chain = [T_LOCAL],
    expected_group = T_LOCAL,
)
results.append(res)

── Test C: Cloud → Local ────────────────────────────────────────────────
  primary   : test/tier-cloud-9151c444
  fallbacks : ['test/tier-local-9151c444']
  expecting : test/tier-local-9151c444


  ✅ HTTP 200 in 6.9s
     x-litellm-model-group       : test/tier-local-9151c444
     x-litellm-attempted-fallbacks: 1
     response body model          : ollama_chat/llama3.2:3b
     answer                       : 'PONG'


### Test D — Full chain: Free → Hybrid → Cloud → Local

All three intermediate tiers are dead sentinels.  The request must traverse
all of them before landing on the real local Ollama model.

In [9]:
print('── Test D: Full chain Free → Hybrid → Cloud → Local ─────────────────────')
res = run_test(
    name           = 'D: Free→Hybrid→Cloud→Local (full chain)',
    primary        = T_FREE,
    fallback_chain = [T_HYBRID, T_CLOUD, T_LOCAL],
    expected_group = T_LOCAL,
)
results.append(res)

── Test D: Full chain Free → Hybrid → Cloud → Local ─────────────────────
  primary   : test/tier-free-9151c444
  fallbacks : ['test/tier-hybrid-9151c444', 'test/tier-cloud-9151c444', 'test/tier-local-9151c444']
  expecting : test/tier-local-9151c444


  ✅ HTTP 200 in 44.5s
     x-litellm-model-group       : test/tier-local-9151c444
     x-litellm-attempted-fallbacks: 1
     response body model          : ollama_chat/llama3.2:3b
     answer                       : 'PONG'


---
## Phase 3 — Config audit

Checks whether the running LiteLLM instance has the correct cross-tier fallback
config active, or whether stale DB-stored config is overriding `litellm_config.yaml`.

### Background: the DB-override problem

With `STORE_MODEL_IN_DB=True`, LiteLLM stores model registrations in PostgreSQL.
If the DB also contains **router_settings** (fallback chains) from an older version
of the config, those override the YAML on restart — even if the YAML was corrected.

Symptom: error messages contain legacy group names like `hybrid-tier`, `local-tier`.

Fix: `docker compose restart litellm` — LiteLLM rewrites router_settings from the
YAML on a clean start, purging stale DB config.

In [10]:
print('── Phase 3: Config audit ─────────────────────────────────────────────────')

EXPECTED_FALLBACKS = {
    'free/chat':     ['hybrid/chat'],
    'free/code':     ['hybrid/code'],
    'free/reason':   ['hybrid/reason'],
    'free/fast':     ['hybrid/fast', 'hybrid/chat'],
    'hybrid/chat':   ['free/chat',    'cloud/fast',   'cloud/chat'],
    'hybrid/code':   ['free/code',    'cloud/code',   'cloud/chat'],
    'hybrid/reason': ['free/reason',  'cloud/reason', 'cloud/chat'],
    'hybrid/fast':   ['free/fast',    'cloud/fast'],
    'cloud/chat':    ['free/chat',    'hybrid/chat'],
    'cloud/fast':    ['free/fast',    'free/chat',    'hybrid/chat'],
    'cloud/code':    ['free/code',    'hybrid/code'],
    'cloud/reason':  ['free/reason',  'hybrid/reason'],
    'local/chat':    ['hybrid/chat',  'free/chat',    'cloud/fast'],
    'local/code':    ['hybrid/code',  'free/code',    'cloud/code'],
    'local/reason':  ['hybrid/reason','free/reason',  'cloud/reason'],
    'local/fast':    ['hybrid/fast',  'free/fast',    'cloud/fast'],
}

# Probe: call a non-existent model group. LiteLLM will return a routing error
# that includes the active fallback map — exposing legacy group names if present.
probe_group = f'probe-audit-nonexistent-{RUN_ID}'
err_msg = ''
try:
    r = chat(probe_group, request_timeout=10)
    if not r.ok:
        try:
            err_msg = r.json().get('error', {}).get('message', r.text)
        except Exception:
            err_msg = r.text
except Exception as e:
    err_msg = str(e)

LEGACY_NAMES = ['hybrid-tier', 'local-tier', 'cloud-tier', 'free-tier']
stale = [n for n in LEGACY_NAMES if n in err_msg]

if stale:
    print('❌ STALE FALLBACK CONFIG DETECTED')
    print(f'   Legacy group names active: {stale}')
    print()
    print('   Root cause: STORE_MODEL_IN_DB=True — the DB has old router_settings')
    print('               that override litellm_config.yaml on startup.')
    print()
    print('   Fix:  docker compose restart litellm')
    print('         LiteLLM re-reads router_settings from the YAML on a clean start.')
    print()
    print('   Impact: real-tier cross-tier fallbacks (free/chat → hybrid/chat etc.)')
    print('           will NOT work until this is fixed.')
    print('           Phase 2 tests above use per-request fallbacks — unaffected.')
    results.append({'test': 'Phase3/config-audit', 'status': 'FAIL',
                    'group': f'stale names: {stale}', 'fb_count': 'n/a', 'elapsed': 'n/a'})
else:
    print('✅ No legacy group names detected in active fallback config.')
    print()
    print('   Expected fallback chains (from litellm_config.yaml):')
    for src, targets in EXPECTED_FALLBACKS.items():
        arrow = ' → '.join(targets)
        print(f'     {src:<20} → {arrow}')
    results.append({'test': 'Phase3/config-audit', 'status': 'PASS',
                    'group': 'no stale config', 'fb_count': 'n/a', 'elapsed': 'n/a'})

print()
print('Probe error message (for manual inspection):')
print(err_msg[:800])

── Phase 3: Config audit ─────────────────────────────────────────────────
✅ No legacy group names detected in active fallback config.

   Expected fallback chains (from litellm_config.yaml):
     free/chat            → hybrid/chat
     free/code            → hybrid/code
     free/reason          → hybrid/reason
     free/fast            → hybrid/fast → hybrid/chat
     hybrid/chat          → free/chat → cloud/fast → cloud/chat
     hybrid/code          → free/code → cloud/code → cloud/chat
     hybrid/reason        → free/reason → cloud/reason → cloud/chat
     hybrid/fast          → free/fast → cloud/fast
     cloud/chat           → free/chat → hybrid/chat
     cloud/fast           → free/fast → free/chat → hybrid/chat
     cloud/code           → free/code → hybrid/code
     cloud/reason         → free/reason → hybrid/reason
     local/chat           → hybrid/chat → free/chat → cloud/fast
     local/code           → hybrid/code → free/code → cloud/code
     local/reason         → hyb

---
## Cleanup — delete all test models

Always run this cell, even if earlier cells failed.  Safe to re-run.

In [11]:
print(f'── Cleanup ({len(_registered)} models to delete) ─────────────────────────────────────')

ok_count = fail_count = 0
for entry in _registered:
    if delete_model(entry):
        print(f'  🗑  {entry["name"]}  ({entry["model_id"]})')
        ok_count += 1
    else:
        fail_count += 1

print(f'\nDeleted {ok_count}, failed {fail_count}')

# Verify no test models remain in the model list
try:
    all_models = requests.get(f'{LITELLM_URL}/v1/models', headers=H, timeout=10).json().get('data', [])
    leftover   = [m['id'] for m in all_models if RUN_ID in m.get('id', '')]
    if leftover:
        print(f'\n⚠  {len(leftover)} test model(s) still listed in /v1/models:')
        for m in leftover:
            print(f'   {m}')
        print('\n  Manual delete:')
        print(f'  curl -X POST {LITELLM_URL}/model/delete '
              f'-H "Authorization: Bearer $LITELLM_API_KEY" '
              f"-d '{{\"id\": \"<model_id>\"}}' ")
    else:
        print('\n✅ No test models remain — cleanup verified.')
except Exception as e:
    print(f'\n⚠  Could not verify via /v1/models: {e}')

── Cleanup (6 models to delete) ─────────────────────────────────────
  🗑  test/tier-free-9151c444  (689799ed-db17-47fd-bba7-a708ae1c330c)
  🗑  test/tier-hybrid-9151c444  (7e1b0083-99a3-4a47-97d4-1381c6847861)
  🗑  test/tier-cloud-9151c444  (236b371d-4e25-4282-af35-962d4f498686)
  🗑  test/tier-local-9151c444  (26717e31-2404-4893-8f21-7eb6810fbc7c)
  🗑  test/tier-hybrid-live-9151c444  (e9c5cd01-72ea-4c45-acc6-0b5025b3535c)
  🗑  test/tier-cloud-live-9151c444  (bf1f6061-df17-4d46-8ecc-da11afebf3eb)

Deleted 6, failed 0

✅ No test models remain — cleanup verified.


---
## Results summary

In [12]:
ICON = {'PASS': '✅', 'WARN': '⚠️ ', 'FAIL': '❌', 'ERROR': '💥'}

print(f'══ Tier Fallback Test Results  run={RUN_ID} ══')
print(f'{"Test":<48} {"Status":<8} {"FB#":<5} {"Time":<7} Serving group / note')
print('─' * 115)

passed = warned = failed = 0
for r in results:
    icon = ICON.get(r['status'], '?')
    if   r['status'] == 'PASS':  passed  += 1
    elif r['status'] == 'WARN':  warned  += 1
    else:                        failed  += 1
    print(f"{r['test']:<48} {icon} {r['status']:<6} "
          f"{r.get('fb_count',''):<5} {r.get('elapsed','n/a'):<7} "
          f"{r.get('group','')[:55]}")

total = passed + warned + failed
print('─' * 115)
print(f'Total {total}  |  ✅ PASS {passed}  |  ⚠️  WARN {warned}  |  ❌/💥 FAIL/ERROR {failed}')

if warned or failed:
    print()
    print('Diagnostic notes:')
    print('  WARN  — request succeeded but PASS criteria not fully met (check group name / fb_count)')
    print('  FAIL  — HTTP error or exception; check the cell output above for the full error')
    print()
    print('  Phase1/cloud FAIL  → Gemini/cloud-fast quota exhausted — expected on free tier')
    print('  Phase2 WARN/FAIL   → per-request fallback did not fire or wrong group served')
    print('  Phase3 FAIL        → stale DB router config; fix: docker compose restart litellm')

══ Tier Fallback Test Results  run=9151c444 ══
Test                                             Status   FB#   Time    Serving group / note
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Phase1/local                                     ✅ PASS   0     6.7s    local/fast
Phase1/free                                      ✅ PASS   0     1.9s    free/fast
Phase1/hybrid                                    ✅ PASS   0     0.2s    hybrid/fast
Phase1/cloud                                     ✅ PASS   0     1.4s    cloud/fast
A: Free→Hybrid                                   ✅ PASS   1     7.1s    test/tier-hybrid-live-9151c444
B: Hybrid→Cloud                                  ✅ PASS   1     7.4s    test/tier-cloud-live-9151c444
C: Cloud→Local                                   ✅ PASS   1     6.9s    test/tier-local-9151c444
D: Free→Hybrid→Cloud→Local (full chain)          ✅ PASS   1     44.5s   test/tier-local-9151c444
Phase3/config